# **Load Base Model and Apply LoRA Fine-Tuned Weights**

In [1]:
# Install dependencies (if not done)
!pip install unsloth peft accelerate sentencepiece transformers==4.56.2 --quiet

# Imports
from unsloth import FastVisionModel
from PIL import Image
import torch

# Load the base model
base_model_id = "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit"  # base model
model, tokenizer = FastVisionModel.from_pretrained(
    base_model_id,
    load_in_4bit=True,            # 4-bit reduces memory
    use_gradient_checkpointing="unsloth",
)

# Merge your LoRA adapter (from Hugging Face or Google Drive)
lora_repo_id = "AIOmarRehan/Llama-3.2-11B-Vision-LoRA-on-Astronomy-with-Unsloth"
model = FastVisionModel.get_peft_model(model, lora_adapter=lora_repo_id)
model.eval()
FastVisionModel.for_inference(model)  # switch to inference mode

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

Unsloth: Making `model.base_model.model.model.vision_model.transformer` require gradients


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MllamaForConditionalGeneration(
      (model): MllamaModel(
        (vision_model): MllamaVisionModel(
          (patch_embedding): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), padding=valid, bias=False)
          (gated_positional_embedding): MllamaPrecomputedPositionEmbedding(
            (tile_embedding): Embedding(9, 8197120)
          )
          (pre_tile_positional_embedding): MllamaPrecomputedAspectRatioEmbedding(
            (embedding): Embedding(9, 5120)
          )
          (post_tile_positional_embedding): MllamaPrecomputedAspectRatioEmbedding(
            (embedding): Embedding(9, 5120)
          )
          (layernorm_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (layernorm_post): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (transformer): MllamaVisionEncoder(
            (layers): ModuleList(
              (0-31): 32 x MllamaVisionEncoderLayer(
         

# **Fast testing with an image and text as a question**

In [2]:
# Load your test image
from google.colab import files
from PIL import Image

uploaded = files.upload()
image_path = list(uploaded.keys())[0]
image = Image.open(image_path).convert("RGB")

# Prepare a text prompt
prompt = "Describe accurately what you see in this image."

# Apply chat template (prepares text + vision tokens)
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},  # PIL Image
            {"type": "text", "text": prompt}
        ]
    }
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

# Tokenize for model (pass images as first positional argument)
inputs = tokenizer(
    image,          # first positional argument: image
    input_text,     # second positional argument: text
    add_special_tokens=False,
    return_tensors="pt"
).to("cuda")

# Generate output
from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

with torch.no_grad():
    _ = model.generate(
        **inputs,
        streamer=text_streamer,
        max_new_tokens=128,
        temperature=1.2
    )

Saving d41586-019-01981-2_16846332.jpg to d41586-019-01981-2_16846332.jpg
This image depicts the Perseverance rover situated on a barren, dusty landscape, showcasing its rugged terrain and rocks. The rover's arm is positioned in the center of the image, with its head turned towards the right side. The surrounding landscape is characterized by sandy dunes, rock formations, and sparse vegetation.

The image is captured in sepia tones, with a faint orange haze visible in the distance. The overall environment appears to be a Mars-like planet, suggesting that the image was taken on another planet or in space. The image conveys a sense of desolation and isolation, with the rover being the only visible feature in the


# **Use the model with Gradio**

In [3]:
import gradio as gr
from PIL import Image
import torch

def chat_with_image(prompt, img):
    if img is None:
        return "Please upload an image!"

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

    inputs = tokenizer(
        img,
        input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=1.5
        )

    # Only the assistant response
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Optional: remove "system" and "user" if present
    if "assistant" in generated_text:
        generated_text = generated_text.split("assistant")[-1].strip()

    return generated_text

# Bigger textbox (lines=15) and only model output
iface = gr.Interface(
    fn=chat_with_image,
    inputs=[gr.Textbox(label="Ask something"), gr.Image(type="pil", label="Upload an image")],
    outputs=gr.Textbox(label="Model Response", lines=15),
    title="AstroVision Chat",
    description="Ask the model about your image. Llama-VL LoRA in action!",
    allow_flagging="never"
)

iface.launch()

# You can adjust the settings based on your needs or runtime constraints—there’s always a trade-off between these tweaks and output quality.

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://10ab365026dddf2783.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
